# TT1 - MDM UBA - 2025

**Tariff classification using NLP**

By Santiago Tedoldi

## Training a DistiltBERT for classification

In [1]:
# Dependencies
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re
from typing import Sequence, Optional, Dict, Any, Tuple


### Raw dataset

In [2]:
colspecs = [(0, 6), (6, None)]
data_type = {'HS06': str}
df = pd.read_fwf('data/raw_data_HScodes_desc.txt',
                 colspecs=colspecs, header=None,
                 names=['HS06', 'GOODS_DESCRIPTION'],
                 dtype=data_type)

### Quick EDA

null and duplicated samples

dropping duplicates

analyzing tops and bottoms regarding frequencies

In [3]:
# Quick EDA
print("=== Quick EDA ===")

# Add HS02 (chapter) and HS04 (heading)
df['HS04'] = df['HS06'].str[:4]
df['HS02'] = df['HS06'].str[:2]

print("Nulls per column:")
print(df.isnull().sum(), "\n")

print("Duplicate rows:", df.duplicated().sum(), "\n")

# Function to build and display freq tables
def freq_table(col, name):
    vc      = df[col].value_counts().rename('count')
    rel     = df[col].value_counts(normalize=True).rename('rel_freq')
    cum     = rel.cumsum().rename('cum_freq')
    summary = pd.concat([vc, rel, cum], axis=1)
    summary['rel_freq'] = (summary['rel_freq'] * 100).round(2).astype(str) + '%'
    summary['cum_freq'] = (summary['cum_freq'] * 100).round(2).astype(str) + '%'

    print(f"## Samples per {name} ({col})\n")
    print("### Top 10")
    print(summary.head(10).to_markdown(), "\n")
    print("### Bottom 10")
    print(summary.tail(10).to_markdown(), "\n")

# Dropping duplicates
df.drop_duplicates(inplace=True)

# Chapter-level (HS02)
freq_table('HS02', 'chapter')

# Heading-level (HS04)
freq_table('HS04', 'heading')

# Subheading-level (HS06)
freq_table('HS06', 'subheading')

=== Quick EDA ===
Nulls per column:
HS06                 0
GOODS_DESCRIPTION    0
HS04                 0
HS02                 0
dtype: int64 

Duplicate rows: 232220 

## Samples per chapter (HS02)

### Top 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     84 |   54901 | 20.5%      | 20.5%      |
|     85 |   33571 | 12.54%     | 33.04%     |
|     87 |   28476 | 10.63%     | 43.67%     |
|     73 |   16173 | 6.04%      | 49.71%     |
|     39 |   12218 | 4.56%      | 54.28%     |
|     90 |   11611 | 4.34%      | 58.61%     |
|     82 |    7972 | 2.98%      | 61.59%     |
|     94 |    7921 | 2.96%      | 64.55%     |
|     40 |    7526 | 2.81%      | 67.36%     |
|     83 |    4285 | 1.6%       | 68.96%     | 

### Bottom 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     41 |      22 | 0.01%      | 99.96%     |
|     81 |      19 | 0.01%      | 99.97%     |
|     45 |      19 | 0

In [4]:
df

,HS06,GOODS_DESCRIPTION,HS04,HS02
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84
2,844399,LCD ASSEMBLY,8443,84
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84
4,630900,USED HANDBAGS AND WALLETS,6309,63
...,...,...,...,...
499959,854239,PCB OPTIONAL ADD. KROPT V4.0 (NEW OUT PUT CARD),8542,85
499961,842091,CYLINDER (SDA80*10F003000001A),8420,84
499970,830249,BEOTIC DEVICE,8302,83
499981,901180,COMPOUND BINOCULAR MICROSCOPE,9011,90


Merging with HS06 nomenclature

In [5]:
df_hs06 = pd.read_csv('data/hs06_full_eng.csv', index_col='hs06', 
                      dtype={'hs06': str, 'full_eng': str},
                      usecols=['hs06', 'full_eng'])

In [6]:
# top 5 rows in HS06 nomemclature
print(df_hs06.head(5).to_markdown(), "\n")

# bottom 5 rows in HS06 nomemclature
print(df_hs06.tail(5).to_markdown(), "\n")

|   hs06 | full_eng                                                                              |
|-------:|:--------------------------------------------------------------------------------------|
| 010120 | Live horses, asses, mules and hinnies. && - Horses :                                  |
| 010121 | Live horses, asses, mules and hinnies. && - Horses : && -- Pure-bred breeding animals |
| 010129 | Live horses, asses, mules and hinnies. && - Horses : && -- Other                      |
| 010130 | Live horses, asses, mules and hinnies. && - Asses                                     |
| 010190 | Live horses, asses, mules and hinnies. && - Other                                     | 

|   hs06 | full_eng                                                                                                                                                                                                                                             |
|-------:|:------------------------------------

In [7]:
df = pd.merge(df, df_hs06,how='left', left_on='HS06', right_on='hs06')

In [8]:
print("Nulls per column:")
print(df.isnull().sum()/len(df), "\n")

Nulls per column:
HS06                 0.000000
GOODS_DESCRIPTION    0.000000
HS04                 0.000000
HS02                 0.000000
full_eng             0.045381
dtype: float64 



There are 4.5 % of goods with no HS full_eng available

They may are not updated codes

### Preprocessing of text

In [9]:
stop_words = {'of', 'or', 'and', 'for', 'than', 'the', 'in', 'with', 'to', 'but', 'by'
             , 'whether', 'on', 'its', 'an', 'their', 'at', 'this', 'which', 'from'
             , 'as', 'be', 'is'}
alphabet_pattern = re.compile(r'[^a-zA-Z]')
alphabet_number_pattern = re.compile(r'[^a-zA-Z0-9]')
remove_pattern = re.compile(r'[\;\,\)\(\[\]\:]')


def refine_text_func(text):
    text = text.lower()
    text = ' '.join([w for w in text.split() if w not in stop_words])
    alphabet = re.sub(alphabet_pattern, ' ', text)
    alphabet_number = re.sub(alphabet_number_pattern, ' ', text)
    remove = re.sub(remove_pattern, ' ', text)
    result = ' '.join([text, alphabet, alphabet_number, remove])
    return result

In [10]:
df['PREPRO_DESCRIPTION'] = df['GOODS_DESCRIPTION'].apply(lambda x: refine_text_func(x))

### N-gram generation

In [11]:
def create_ngram_data(text, ngram_value=2):
    text_list = text.split()
    ngram_list = list(zip(*[text_list[i:] for i in range(ngram_value)]))
    result = []
    for n_data in ngram_list:
        result.append('_'.join(n_data))
    return ' '.join(result)

create_ngram_data('LIVE BREEDING FARM HORSE')

'LIVE_BREEDING BREEDING_FARM FARM_HORSE'

In [12]:
df['NGRAM_DESCRIPTION'] = df['PREPRO_DESCRIPTION'].apply(lambda x: create_ngram_data(x))

In [13]:
df.head()

,HS06,GOODS_DESCRIPTION,HS04,HS02,full_eng,PREPRO_DESCRIPTION,NGRAM_DESCRIPTION
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27,Petroleum oils and oils obtained from bitumino...,brake fluid dot 4 50x200ml brake fluid dot ...,brake_fluid fluid_dot dot_4 4_50x200ml 50x200m...
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84,Machinery for working rubber or plastics or fo...,plastic injection mould model 21a 110g dsm1010...,plastic_injection injection_mould mould_model ...
2,844399,LCD ASSEMBLY,8443,84,Printing machinery used for printing by means ...,lcd assembly lcd assembly lcd assembly lcd ass...,lcd_assembly assembly_lcd lcd_assembly assembl...
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84,"Ball or roller bearings. && - Other, including...",bearing 22238 kcaw33c3 brand mcb bearing ...,bearing_22238 22238_kcaw33c3 kcaw33c3_brand br...
4,630900,USED HANDBAGS AND WALLETS,6309,63,NaN,used handbags wallets used handbags wallets us...,used_handbags handbags_wallets wallets_used us...


### DistilBERT model training

Iteraring to measure stability

In [14]:
import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from transformers import DistilBertModel
from transformers import DistilBertTokenizerFast

Dataset & DataLoader preparation

In [15]:
# Sampling for testing the pipeline
# df = df.sample(frac=0.01, random_state=42)

Pre-tokenizacion

In [16]:
from distilbert_utils import TokenizedDataset

Tokenizer

In [17]:
# Load the tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

Model class

In [18]:
from distilbert_utils import HSClassifier

Training utils

In [19]:
from tqdm.auto import tqdm
from distilbert_utils import train_epoch, eval_model

Hardware

In [20]:
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
NVIDIA GeForce RTX 3060 Laptop GPU


Evaluation utils

In [21]:
from distilbert_utils import predict_and_evaluate

Iterarion definitions

In [22]:
fraction = 0.05
iterations = 10

# min_val = 0
# max_val = 999999999
# random_seed = random.randint(min_val, max_val)

# seeds = []

# for iter in range(iterations):
#     seed = random.randint(min_val, max_val)
#     seeds.append(seed)

# print("Random seeds for each iteration:")
# print(seeds)  

out_dir = "results/distilbert/"
os.makedirs(out_dir, exist_ok=True)

Config columns

In [23]:
target_col = 'HS04'

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

Config dataset

In [24]:
max_length = 300
loader_batch_size = 32
shuffle = True

label_dir = "models/labels/"
os.makedirs(label_dir, exist_ok=True)

Config train

In [25]:
lr=2e-5

Iteration function

In [26]:
from distilbert_utils import iterative_training

#### Transfer learning

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

A- Raw descriptions

In [ ]:
train_type = "tf" # transfer learning - fixed encoder
fine_tune = False
max_epochs = 20

text_col = raw_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    verbose=True
)

=== DBERT_tf_GOODS_DESCRIPTION_HS04 ===

=== Iteration 1/10 seed 964706610 ===
Model name: DBERT_tf_GOODS_DESCRIPTION_HS04_seed964706610
Training on cuda
Epoch 1/20
----------
Model is training on: cuda:0


Training:   0%|          | 0/7961 [00:00<?, ?it/s]

B- Preproced descriptions

In [ ]:
text_col = prepro_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    verbose=True
)

=== DBERT_tf_PREPRO_DESCRIPTION_HS04 ===

=== Iteration 1/2 seed 826661138 ===
Model name: DBERT_tf_PREPRO_DESCRIPTION_HS04_seed826661138
Training on cuda
Epoch 1/30
----------
Model is training on: cuda:0


Training:   0%|          | 0/80 [00:00<?, ?it/s]


Train loss 6.3325 acc 0.0035 top5 0.0181
Val   loss 6.1232 acc 0.0301 top5 0.0526
Epoch completed in 0.53 minutes.

Epoch 2/30
----------
Model is training on: cuda:0


Training:   0%|          | 0/80 [00:00<?, ?it/s]


Train loss 6.1516 acc 0.0236 top5 0.0538
Val   loss 5.9996 acc 0.0602 top5 0.0977
Epoch completed in 0.57 minutes.

Epoch 3/30
----------
Model is training on: cuda:0


...


Train loss 5.8656 acc 0.0719 top5 0.1202
Val   loss 5.8114 acc 0.1053 top5 0.1654
Epoch completed in 0.57 minutes.

Epoch 5/30
----------
Model is training on: cuda:0


Training:   0%|          | 0/80 [00:00<?, ?it/s]


Train loss 5.7115 acc 0.0860 top5 0.1473
Val   loss 5.7131 acc 0.1128 top5 0.1955
Epoch completed in 0.62 minutes.

Epoch 6/30
----------
Model is training on: cuda:0


Training:   0%|          | 0/80 [00:00<?, ?it/s]

KeyboardInterrupt: 

C- Preproced + N-gram descriptions

In [ ]:
text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    verbose=True
)

### Fine-tuned model

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

A- Raw descriptions

In [ ]:
train_type = "ft" # fine tuned
fine_tune = True
max_epochs = 7

text_col = raw_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    verbose=True
)

B- Preproced descriptions

In [ ]:
text_col = prepro_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    verbose=True
)

C- Preproced + N-gram descriptions

In [ ]:
text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    verbose=True
)

### Partial fine-tuned


Fine-tuning last 2 layers

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

A- Raw descriptions

In [ ]:
train_type = "pft" # partial fine tuned
fine_tune = True
layers_to_finetune = 2
max_epochs = 10

text_col = raw_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    n_finetune_layers=layers_to_finetune
    verbose=True
)

B- Preproced descriptions

In [ ]:
text_col = prepro_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    n_finetune_layers=layers_to_finetune
    verbose=True
)

C- Preproced + N-gram descriptions

In [ ]:
text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    max_epochs=max_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    # seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    n_finetune_layers=layers_to_finetune
    verbose=True
)